## Quality controlling 

A fairly standard start to QC. This is a two-step that relies on bbduk for a quick first-pass good enough for most things and iu-filter-minoche to clean up for maximum quality. OK to skip the iu-filter-minoche step in certain circumstances

Are you in a screen right now? you might want to be.

In [ ]:
import pandas as pd
import os
import re

### First step: bbduk

Make sure to edit the `sed` lines to match your samples specifically.

Recommend first commenting off the `bbduk.sh ...` line and uncommenting the `echo $pref` line to test first before running it

In [ ]:
%%bash 

mkdir -p bbduk_qc   # make bbduk_qc here ahead of time so you don't get errors

for r1 in raw/*_R1_*; do   # easeir for cleaning to keep paths short
pref=$(basename $r1 | sed 's/_ANME.*$//')  # all you need if just short path
r2=$(echo $r1 | sed 's/_R1_/_R2_/')
#echo $pref

# if using short paths then bbduk needs to include the bbduk_qc in out path
bbduk.sh in1=$r1 in2=$r2 ref=/export/data1/sw/anaconda3-2019.07/envs/read_mapping/opt/bbmap-38.87-0/resources/adapters.fa t=6 ktrim=r k=23 mink=11 hdist=1 tpe tbo qtrim=r trimq=10 maq=10 out1=bbduk_qc/${pref}_R1.fastq.gz out2=bbduk_qc/${pref}_R2.fastq.gz

done

What the above cell does:

1. generate a prefix off part of the filename
2. make subfolder bbduk under working directory
3. genete read 2 path off of read 1
**Note**
* R1 and R2 need to have same unique sample name to be paired
* Check if expansion is fastq.gz or fq.gz
* r=trim to the right
* trimq=  Regions with average quality BELOW this will be trimmed

### iu-filter-minoche to polish
This step creates create config file for [illumina-utils](https://github.com/merenlab/illumina-utils?tab=readme-ov-file#config-file-format) based on [this paper](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0066643)

Overarching pattern is to list the contents of bbduk_qc and group them into samples. Note that this is often a good time to combine fastq's that are arbitrarily split across lanes at the sequencer. For examples patterns like: 

```
samp1_lane1_R1.fq.gz
samp1_lane1_R2.fq.gz
samp1_lane2_R1.fq.gz
samp1_lane2_R2.fq.gz
samp2_lane1_R1.fq.gz
...
```

should become in the config file

```
pair_1 = samp1_lane1_R1.fq.gz, samp1_lane2_R1.fq.gz
pair_2 = samp1_lane1_R2.fq.gz, samp1_lane2_R2.fq.gz
```

Edit the `re.sub` parts until it is correct.

**IMPORTANT** Make sure to edit input_directory and output_directory in the `general_config` section to point to the specific directory.

In [ ]:
!mkdir -p iu_filter_minoche

In [ ]:

fastqs = os.listdir('bbduk_qc')
# samples = [x.split("_")[:][0] for x in fastqs]
samples = [x.split("_R")[:][0] for x in fastqs]
# get rid of lane-specific identification so it will actually collapses
samples = [re.sub("_CKDN.*_L[0-9]*","", x) for x in samples]
#check no sample has naming of _R!!
samples = {s:[f for f in fastqs if f.startswith(s)] for s in samples}
print(samples)  # not a bad idea to print and check what it plans to collapse or not


general_config = """[general]
project_name = SAMPLE_OUT
researcher_email = 'dutter@caltech.edu'
input_directory = /export/data1/data/metagenomes/ANME_mats/bbduk_qc
output_directory = /export/data1/data/metagenomes/ANME_mats/iu_filter_minoche


"""
        
for samp in samples.keys():
    with open("iu_filter_minoche/config_" + samp + ".txt", 'w') as f:
        f.write(re.sub("SAMPLE_OUT", samp, general_config))
        f.write("[files]\n")
        r1_files = sorted([f for f in samples[samp] if "_R1" in f])
        r2_files = sorted([f for f in samples[samp] if "_R2" in f])
        f.write("pair_1 = " + ", ".join(r1_files) + "\n")
        f.write("pair_2 = " + ", ".join(r2_files))


These config files need to be run in `conda activate anvio-9` environment (any `anvio` environment)

Best done in a screen as it can take awhile.

Then run (in parallel if enough threads availble) with
```
# sequential version
for conf in config*.txt; do iu-filter-quality-minoche $conf; done

# OR parallel version:
# for conf in config*.txt; do (iu-filter-quality-minoche $conf)& done
